## Dependencies


In [2]:
import json
import datetime
import mermaid
import subprocess
import arviz as az
import polars as pl
import numpy as np
import pymc as pm
from pathlib import Path
from matplotlib import pyplot as plt, ticker, dates, patheffects

In [3]:
az.rcParams["stats.ci_prob"] = 0.89
plt.rcParams["figure.dpi"] = 512

PRE_BEDTIME_SCREENTIME_WINDOW = 2  # hours

## Retrieving Data


### Apple Screen Time


In [23]:
def read_laptop_log():
    !cp "$HOME/Library/Application Support/Knowledge/knowledgeC.db" "data/"
    !cp "$HOME/Library/Application Support/Knowledge/knowledgeC.db-wal" "data/"
    !sqlite3 "data/knowledgeC.db" "PRAGMA wal_checkpoint(TRUNCATE)" > /dev/null 2>&1
    device_id = !system_profiler SPHardwareDataType 2>/dev/null | awk '/Hardware UUID/{print $3}'
    database_query = """
    SELECT
        ZSOURCE.ZDEVICEID as "device_id",
        CAST(ZOBJECT.ZSTARTDATE + 978307200 AS INTEGER) as "timestamp_start",
        ZOBJECT.ZSECONDSFROMGMT / 3600 AS "timezone_offset_hr",
        ZOBJECT.ZVALUESTRING as "app",
        CAST(ZOBJECT.ZENDDATE - ZOBJECT.ZSTARTDATE AS INTEGER) as "duration_seconds",
        ZOBJECT.ZSTREAMNAME as "stream",
        ZMODEL as "source"
    FROM
        ZOBJECT
        LEFT JOIN ZSTRUCTUREDMETADATA ON ZOBJECT.ZSTRUCTUREDMETADATA = ZSTRUCTUREDMETADATA.Z_PK
        LEFT JOIN ZSOURCE ON ZOBJECT.ZSOURCE = ZSOURCE.Z_PK
        LEFT JOIN ZSYNCPEER ON ZSOURCE.ZDEVICEID = ZSYNCPEER.ZDEVICEID
    ORDER BY
        ZOBJECT.ZSTARTDATE
    """
    return (
        pl.concat(
            pl.read_database_uri(
                query=database_query,
                uri=f"sqlite://{event_log_filepath}",
            )
            for event_log_filepath in Path("data").glob("knowledgeC*")
        )
        .unique(
            keep="first",
        )
        .filter(
            pl.col("stream").eq("/app/usage"),
        )
        .with_columns(
            pl.col("device_id").fill_null(device_id[0]),
            pl.from_epoch(
                pl.col("timestamp_start").add(pl.col("timezone_offset_hr").mul(3600))
            ),
            pl.duration(hours=pl.col("timezone_offset_hr")).alias("timezone_offset"),
        )
        .insert_column(
            0,
            pl.lit("MacBook Pro").alias("device_type"),
        )
    )

def read_phone_log():
    !aw-import-screentime events preview --since=60d --limit=0 > data/events.json 2>/dev/null
    return (
        pl.concat(
            pl.read_json(event_log_filepath)
            .explode("events")
            .drop_nulls("events")
            .unnest("events")
            .unnest("data")
            .drop("files_scanned")
            for event_log_filepath in Path("data").glob("events*")
        )
        .unique(
            keep="first",
        )
        .with_columns(
            pl.col("timestamp")
            .str.to_datetime(time_zone="UTC").dt.convert_time_zone(time_zone="Europe/Copenhagen").dt.replace_time_zone(None).alias("timestamp_start"),
        )
        .insert_column(
            0,
            pl.lit("iPhone 15 Plus").alias("device_type"),
        )
    )


def retrieve_merged_screentime_data():
    screentime = pl.concat(
        [
            read_laptop_log(),
            read_phone_log(),
        ],
        how="diagonal_relaxed",
    ).with_columns(
        pl.duration(seconds=pl.col("duration_seconds")).alias("screentime"),
        pl.col("timestamp_start")
        .add(pl.duration(seconds=pl.col("duration_seconds")))
        .alias("timestamp_end"),
        pl.col("title").fill_null(pl.col("app").str.split(".").list.last()),
    ).insert_column(
        0,
        pl.col("timestamp_start").dt.date().alias("date"),
    ).select(
        "date",
        "device_type",
        "timestamp_start",
        "timestamp_end",
        # "timezone_offset",
        "screentime",
        "app",
        # "title",
    ).sort("timestamp_start", descending=True)

    return screentime


screentime = pl.read_parquet(
    screentime_export_path := "data/screentime.parquet"
)

if (datetime.date.today() - screentime.head(1)["date"].item()).days:
    screentime = pl.concat([
        screentime,
        retrieve_merged_screentime_data(),
    ], how="diagonal_relaxed").unique(subset=("device_type", "timestamp_start", "timestamp_end",), keep="first").sort("timestamp_start", descending=True)
    screentime.write_parquet(screentime_export_path)

screentime = screentime.filter(
    pl.col("date").ne(pl.col("date").last()),
    pl.col("app").ne("com.apple.InCallService"),
)

screentime.select(pl.exclude("app"))

date,device_type,timestamp_start,timestamp_end,screentime
date,str,datetime[μs],datetime[μs],duration[μs]
2026-05-24,"""MacBook Pro""",2026-05-24 08:20:27,2026-05-24 08:20:30,3s
2026-05-24,"""MacBook Pro""",2026-05-24 08:19:38,2026-05-24 08:20:13,35s
2026-05-24,"""iPhone 15 Plus""",2026-05-24 08:04:21.976,2026-05-24 08:05:10.137881,48s 161881µs
2026-05-24,"""iPhone 15 Plus""",2026-05-24 08:04:18.892,2026-05-24 08:04:21.940868,3s 48868µs
2026-05-24,"""iPhone 15 Plus""",2026-05-24 08:03:25.812,2026-05-24 08:04:16.189569,50s 377569µs
…,…,…,…,…
2026-04-02,"""iPhone 15 Plus""",2026-04-02 08:35:01.102,2026-04-02 08:35:07.375395,6s 273395µs
2026-04-02,"""iPhone 15 Plus""",2026-04-02 08:34:51.881,2026-04-02 08:35:01.101789,9s 220789µs
2026-04-02,"""iPhone 15 Plus""",2026-04-02 08:09:24.249,2026-04-02 08:25:57.834632,16m 33s 585632µs


In [5]:
screentime.group_by("device_type", maintain_order=True).agg(
    pl.len().alias("instances"),
    pl.col("screentime").sum(),
    pl.col("timestamp_start").sort().first().dt.date().alias("first observation"),
    pl.col("timestamp_start").sort().last().dt.date().alias("most recent observation"),
    pl.col("timestamp_start").dt.date().n_unique().alias("days observed"),
).sort("screentime", "device_type", descending=True).group_by(
    "device_type", maintain_order=True
).head(
    5
)

device_type,instances,screentime,first observation,most recent observation,days observed
str,u32,duration[μs],date,date,u32
"""MacBook Pro""",17857,11d 8m 39s,2026-04-02,2026-05-24,51
"""iPhone 15 Plus""",9626,6d 17h 54m 22s 114794µs,2026-04-02,2026-05-24,53


In [6]:
screentime.group_by(
    pl.col("timestamp_start").dt.date().alias("date"),
    # "device_type",
).agg(
    pl.col("screentime").sum(),
).sort(
    "date",
    # "device_type",
    descending=True,
)

date,screentime
date,duration[μs]
2026-05-24,2m 19s 588318µs
2026-05-23,5h 16m 4s 667118µs
2026-05-22,10h 11m 25s 10315µs
2026-05-21,7h 11m 17s 162998µs
2026-05-20,8h 27m 1s 449115µs
…,…
2026-04-06,7h 17m 15s 625124µs
2026-04-05,8h 31m 28s 424368µs
2026-04-04,7h 16m 23s 413548µs


In [7]:
# print(
#     f"""{screentime["screentime"].dt.total_hours(fractional=True).sum():.0f} hours of screentime across {screentime["date"].n_unique()} days (median|mean app usage duration: {screentime["screentime"].dt.total_seconds(fractional=True).median():.1f}|{screentime["screentime"].dt.total_seconds(fractional=True).mean():.1f} seconds)"""
# )

In [8]:
def merge_screentime_spans(screentime):
    """
    merging screentime spans that overlap (multiple devices used simultaneously) to avoid overcounting
    """
    return (
        screentime.unpivot(on=["timestamp_start", "timestamp_end"], value_name="ts")
        .with_columns(
            pl.when(pl.col("variable").eq("timestamp_start"))
            .then(1)
            .otherwise(-1)
            .alias("delta")
        )
        .sort("ts")
        .with_columns(pl.col("delta").cum_sum().alias("active_devices"))
        .with_columns(
            (pl.col("active_devices").sub(pl.col("delta")).eq(0)).alias("is_start")
        )
        .filter(pl.col("active_devices").eq(0).or_(pl.col("is_start")))
        .with_columns(pl.col("ts").shift(-1).alias("end"))
        .filter(pl.col("is_start"))
        .select(
            pl.col("ts").dt.date().alias("date"),
            pl.col("ts").alias("timestamp_start"),
            pl.col("end").alias("timestamp_end"),
            pl.col("end").sub(pl.col("ts")).alias("screentime"),
        )
    )


screentime_spans = merge_screentime_spans(screentime)

screentime_spans.group_by("date").agg(pl.col("screentime").sum()).sort(
    "date",
    descending=True,
)

date,screentime
date,duration[μs]
2026-05-24,2m 19s 588318µs
2026-05-23,4h 44m 14s 941763µs
2026-05-22,9h 20m 43s 620083µs
2026-05-21,6h 57m 31s 885829µs
2026-05-20,8h 6m 35s 971691µs
…,…
2026-04-06,6h 29m 7s 789869µs
2026-04-05,7h 53m 28s 800623µs
2026-04-04,6h 54m 22s 71623µs


## WHOOP


In [9]:
def extract_response(endpoint):
    """
    extract WHOOP API responses and handle pagintion to fully enumerate data
    """
    result = []
    cursor = None
    while cursor != "":
        response = json.loads(
            subprocess.run(
                f"whoopy {endpoint} list --last=2mo"
                + ("" if cursor is None else f" --cursor={cursor}"),
                shell=True,
                capture_output=True,
            ).stdout
        )
        cursor = response.pop("next_token", "")
        result += response[next(iter(response))]
    return result


def handle_whoop_timestamping(dataframe):
    return (
        dataframe.with_columns(
            pl.col("timezone_offset")
            .str.replace("Z", "+00:00", literal=True)
            .str.splitn(by=":", n=2)
            .struct.rename_fields(["offset_hours", "offset_minutes"])
            .struct.unnest()
        )
        .with_columns(
            pl.duration(
                hours=pl.col("offset_hours").cast(int),
                minutes=pl.col("offset_minutes").cast(int),
            ).alias("timezone_offset"),
        )
        .with_columns(
            (
                pl.col(name)
                .str.slice(0, 19)
                .str.to_datetime(time_zone="UTC")
                .alias(f"{name}_UTC")
                for name in ("start", "end", "created_at", "bedtime", "waketime")
            ),
        )
        .with_columns(
            (
                pl.col(f"{name}_UTC")
                .add(pl.col("timezone_offset"))
                .dt.replace_time_zone(None)
                .alias(name)
                for name in ("start", "end", "created_at", "bedtime", "waketime")
            ),
        )
        .drop(
            "timezone_offset",
            "offset_hours",
            "offset_minutes",
        )
    )


def retrieve_merged_whoop_data():
    cycles = (pl.DataFrame(extract_response(endpoint="cycles")).unnest("score")).drop(
        "user_id",
        "score_state",
        "updated_at",
    )

    recoveries = (
        pl.DataFrame(extract_response(endpoint="recovery")).unnest("score")
    ).drop(
        "user_id",
        "score_state",
        "created_at",
        "updated_at",
        "user_calibrating",
    )

    sleeps = (
        pl.DataFrame(extract_response(endpoint="sleep"))
        .unnest("score")
        .unnest("stage_summary")
        .filter(pl.col("nap").not_())
        .with_columns(
            pl.col("start").alias("bedtime"),
            pl.col("end").alias("waketime"),
        )
        .drop(
            "start",
            "end",
            "user_id",
            "score_state",
            "created_at",
            "updated_at",
            "nap",
        )
    )

    merged_physiologicals = (
        handle_whoop_timestamping(
            cycles.join(
                recoveries,
                how="left",
                left_on="id",
                right_on="cycle_id",
            )
            .join(
                sleeps,
                how="left",
                left_on="sleep_id",
                right_on="id",
            )
            .drop(
                "id",
                "sleep_id",
                "cycle_id",
            )
        )
        .insert_column(
            0,
            pl.col("created_at").dt.date().alias("date"),
        )
        .sort("start", descending=True)
        .limit(-1)
        .select(
            "date",
            # pl.col("start").alias("cycle_start"),
            # pl.col("end").alias("cycle_end"),
            # "created_at",
            "bedtime",
            "waketime",
            "recovery_score",
            pl.col("strain").shift(-1).round(2).alias("prev_day_strain"),
            pl.col("hrv_rmssd_milli").round(2).alias("morning_hrv"),
            pl.col("hrv_rmssd_milli").shift(-1).round(2).alias("prev_morning_hrv"),
            "resting_heart_rate",
            "average_heart_rate",
            "respiratory_rate",
            "spo2_percentage",
            "skin_temp_celsius",
            pl.col("sleep_efficiency_percentage").round(2),
            pl.col("sleep_consistency_percentage").round(2),
            pl.col("sleep_performance_percentage").round(2),
            pl.duration(milliseconds=pl.col("total_in_bed_time_milli")).alias(
                "duration_in_bed"
            ),
            pl.col("waketime")
            .sub(pl.col("bedtime"))
            .sub(pl.duration(milliseconds=pl.col("total_awake_time_milli")))
            .alias("duration_asleep"),
            pl.duration(milliseconds=pl.col("total_rem_sleep_time_milli")).alias(
                "duration_in_rem"
            ),
            pl.col("total_rem_sleep_time_milli")
            .truediv(pl.col("total_in_bed_time_milli"))
            .alias("proportion_in_rem"),
            pl.duration(milliseconds=pl.col("total_slow_wave_sleep_time_milli")).alias(
                "duration_in_slowwave"
            ),
            pl.col("total_slow_wave_sleep_time_milli")
            .truediv(pl.col("total_in_bed_time_milli"))
            .alias("proportion_in_slowwave"),
            "sleep_cycle_count",
            pl.col("disturbance_count").alias("sleep_disturbance_count"),
        )
    )
    return merged_physiologicals


merged_physiologicals = pl.read_parquet(
    physiologicals_export_path := "data/physiologicals.parquet"
)

if (datetime.date.today() - merged_physiologicals.head(1)["date"].item()).days:
    merged_physiologicals = retrieve_merged_whoop_data()
    merged_physiologicals.write_parquet(physiologicals_export_path)

merged_physiologicals

date,bedtime,waketime,recovery_score,prev_day_strain,morning_hrv,prev_morning_hrv,resting_heart_rate,average_heart_rate,respiratory_rate,spo2_percentage,skin_temp_celsius,sleep_efficiency_percentage,sleep_consistency_percentage,sleep_performance_percentage,duration_in_bed,duration_asleep,duration_in_rem,proportion_in_rem,duration_in_slowwave,proportion_in_slowwave,sleep_cycle_count,sleep_disturbance_count
date,datetime[μs],datetime[μs],i64,f64,f64,f64,i64,i64,f64,f64,f64,f64,i64,i64,duration[μs],duration[μs],duration[μs],f64,duration[μs],f64,i64,i64
2026-05-24,2026-05-23 23:28:02,2026-05-24 08:02:00,65,6.24,67.94,67.69,56,55,13.632812,95.85714,34.748665,94.25,88,92,8h 33m 57s 931ms,8h 4m 25s 940ms,2h 3m 17s 911ms,0.239896,1h 36m 32s 330ms,0.187831,2,11
2026-05-23,2026-05-22 23:11:30,2026-05-23 07:26:36,79,5.08,67.69,69.34,52,62,13.007812,95.0,34.15867,91.78,91,98,8h 15m 6s 301ms,7h 32m 28s 920ms,2h 31s 520ms,0.243434,1h 58m 32s 410ms,0.239424,8,18
2026-05-22,2026-05-21 23:11:02,2026-05-22 07:42:28,84,10.67,69.34,62.18,52,61,13.066406,96.85,35.173,94.99,83,94,8h 31m 25s 610ms,8h 4m 10s 890ms,1h 17m 1s 210ms,0.150599,1h 50m 32s 340ms,0.216138,7,21
2026-05-21,2026-05-20 23:17:59,2026-05-21 07:13:22,60,4.48,62.18,69.09,53,64,13.242188,95.454544,34.081833,92.38,80,89,7h 55m 22s 490ms,7h 16m 44s 750ms,1h 34m 5s 310ms,0.197925,1h 56m 3s 360ms,0.244136,5,12
2026-05-20,2026-05-19 23:02:34,2026-05-20 07:40:47,85,4.42,69.09,68.25,52,63,13.066406,94.75,34.613667,94.01,75,89,8h 38m 12s 926ms,8h 7m 11s,2h 12m 32s 600ms,0.255769,2h 5m 4s 360ms,0.241353,6,16
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-03-30,2026-03-29 23:36:23,2026-03-30 08:36:40,50,12.53,62.81,64.78,53,61,12.929688,95.02273,34.391003,93.96,76,89,9h 17s 690ms,8h 25m 13s 740ms,1h 49m 32s 330ms,0.202739,2h 23m 1s 420ms,0.264714,8,8
2026-03-29,2026-03-29 01:22:45,2026-03-29 09:00:17,51,4.14,64.78,68.68,52,68,12.832031,96.4375,34.578335,95.72,83,90,7h 37m 32s 13ms,7h 16m 24s 940ms,1h 43m 45s 973ms,0.226795,1h 27m 4s 200ms,0.190303,7,9
2026-03-28,2026-03-28 00:01:55,2026-03-28 08:17:40,55,4.12,68.68,78.44,53,63,13.066406,96.27273,34.664833,91.68,79,90,8h 15m 45s 380ms,7h 32m 10s 940ms,1h 42m 31s 220ms,0.206796,1h 48m 33s 350ms,0.21897,6,16


In [10]:
journal = (
    (
        pl.read_csv("data/journal_entries.csv")
        .with_columns(
            pl.concat_str(
                pl.col("Cycle start time"),
                pl.col("Cycle timezone").str.slice(3, length=None),
            )
            .str.to_datetime(time_zone="Europe/Copenhagen")
            .dt.replace_time_zone(None),
            pl.concat_str(
                pl.col("Cycle end time"),
                pl.col("Cycle timezone").str.slice(3, length=None),
            )
            .str.to_datetime(time_zone="Europe/Copenhagen")
            .dt.replace_time_zone(None),
        )
        .select(
            pl.col("Cycle start time").alias("cycle_start"),
            pl.col("Cycle end time").alias("cycle_end"),
            pl.col("Question text").alias("prompt"),
            pl.col("Answered yes").alias("response").cast(pl.Int32),
            pl.col("Notes").alias("notes"),
        )
    )
    .pivot(
        on="prompt",
        index="cycle_start",
        values="response",
        aggregate_function="first",
    )
    .with_columns()
    .sort("cycle_start", descending=True)
)
merged_physiologicals = merged_physiologicals.join(
    journal,
    how="left",
    left_on="bedtime",
    right_on="cycle_start",
).pipe(
    lambda df: df.select(col for col in df.columns if df[col].null_count() < len(df))
)

## Merging Data Sources


In [11]:
screentime_prediction_range = np.linspace(0, PRE_BEDTIME_SCREENTIME_WINDOW, 5)

screentime_physiologicals = (
    merged_physiologicals.with_columns(
        pl.col("waketime").shift(-1).alias("prev_waketime")
    )
    .join(screentime_spans.drop("date"), how="cross")
    .filter(
        pl.col("timestamp_start")
        .le(pl.col("bedtime"))
        .and_(pl.col("timestamp_start").ge(pl.col("prev_waketime"))),
    )
    .with_columns(
        pl.min_horizontal("timestamp_end", pl.col("bedtime"))
        .sub(
            pl.max_horizontal(
                "timestamp_start",
                pl.col("bedtime").sub(pl.duration(hours=PRE_BEDTIME_SCREENTIME_WINDOW)),
            )
        )
        .clip(lower_bound=pl.duration(seconds=0))
        .alias("clipped_screentime"),
    )
)

daily_screentime_physiology = (
    screentime_physiologicals.group_by(
        "date",
    )
    .agg(
        pl.col("screentime").sum(),
        pl.col("clipped_screentime").sum().alias("screentime_pre_bedtime"),
        pl.col("prev_day_strain").first(),
        pl.col("bedtime").first(),
        pl.col("prev_morning_hrv").first(),
        pl.col("morning_hrv").first(),
        pl.col("sleep_efficiency_percentage").first(),
        pl.col("sleep_disturbance_count").first(),
        pl.col("duration_in_slowwave").first(),
        pl.col("proportion_in_slowwave").first(),
        pl.col("duration_asleep").first(),
        pl.col("Connected with family and/or friends?").first(),
        pl.col("Faced challenges?").first(),
        pl.col("Experienced stress?").first(),
    )
    .sort(
        "date",
        descending=True,
    )
    .with_columns(
        pl.col("screentime_pre_bedtime")
        .truediv(pl.col("screentime"))
        .round(2)
        .alias("proportion_pre_bedtime"),
        pl.col("screentime_pre_bedtime")
        .sub(pl.col("screentime_pre_bedtime").mean())
        .truediv(pl.col("screentime_pre_bedtime").std())
        .alias("Z_screentime_pre_bedtime"),
        pl.col("screentime").dt.total_hours(fractional=True).alias("screentime_hours"),
        pl.col("screentime_pre_bedtime")
        .dt.total_hours(fractional=True)
        .alias("screentime_pre_bedtime_hours"),
        pl.col("duration_asleep")
        .dt.total_hours(fractional=True)
        .alias("duration_asleep_hours"),
    )
    .limit(-1)
    .head(50)
)
daily_screentime_physiology

date,screentime,screentime_pre_bedtime,prev_day_strain,bedtime,prev_morning_hrv,morning_hrv,sleep_efficiency_percentage,sleep_disturbance_count,duration_in_slowwave,proportion_in_slowwave,duration_asleep,Connected with family and/or friends?,Faced challenges?,Experienced stress?,proportion_pre_bedtime,Z_screentime_pre_bedtime,screentime_hours,screentime_pre_bedtime_hours,duration_asleep_hours
date,duration[μs],duration[μs],f64,datetime[μs],f64,f64,f64,i64,duration[μs],f64,duration[μs],i32,i32,i32,f64,f64,f64,f64,f64
2026-05-24,4h 44m 14s 941763µs,40m 2s 642107µs,6.24,2026-05-23 23:28:02,67.69,67.94,94.25,11,1h 36m 32s 330ms,0.187831,8h 4m 25s 940ms,null,null,null,0.14,-0.621462,4.737484,0.667401,8.073872
2026-05-23,9h 20m 43s 620083µs,41m 19s 502102µs,5.08,2026-05-22 23:11:30,69.34,67.69,91.78,18,1h 58m 32s 410ms,0.239424,7h 32m 28s 920ms,null,null,null,0.07,-0.575745,9.34545,0.688751,7.541367
2026-05-22,6h 57m 31s 885829µs,50m 43s 326303µs,10.67,2026-05-21 23:11:02,62.18,69.34,94.99,21,1h 50m 32s 340ms,0.216138,8h 4m 10s 890ms,null,null,null,0.12,-0.240374,6.958857,0.845368,8.069692
2026-05-21,8h 6m 35s 971691µs,26m 43s 560774µs,4.48,2026-05-20 23:17:59,69.09,62.18,92.38,12,1h 56m 3s 360ms,0.244136,7h 16m 44s 750ms,null,null,null,0.05,-1.096769,8.109992,0.445434,7.279097
2026-05-20,8h 5m 27s 833048µs,24m 15s 383145µs,4.42,2026-05-19 23:02:34,68.25,69.09,94.01,16,2h 5m 4s 360ms,0.241353,8h 7m 11s,null,null,null,0.05,-1.184907,8.091065,0.404273,8.119722
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-04-09,7h 32m 12s 279213µs,45m 3s 547377µs,5.17,2026-04-08 23:31:20,74.4,73.54,91.09,14,1h 48m 34s 370ms,0.218694,7h 30m 12s 760ms,1,1,0,0.1,-0.442479,7.536744,0.750985,7.503544
2026-04-08,5h 26m 35s 288155µs,1h 3m 33s 695065µs,11.14,2026-04-07 22:55:01,77.51,74.4,93.13,7,2h 34s 390ms,0.230773,8h 1m 41s 930ms,1,1,0,0.19,0.217854,5.443136,1.05936,8.028314
2026-04-07,6h 29m 7s 789869µs,1h 39m 29s 624692µs,12.42,2026-04-06 23:06:38,65.8,77.51,91.47,11,1h 46m 3s 340ms,0.203259,7h 57m 15s 930ms,1,1,0,0.26,1.500234,6.485497,1.658229,7.954425


## Exploratory Data Analysis


In [12]:
fig, ax = plt.subplots(
    figsize=(9, 4.5),
)

screentime_spans = screentime_spans.filter(
    pl.col("date").is_in(daily_screentime_physiology["date"].implode())
)

for date, data in screentime_spans.group_by(
    "date",
):
    ax.hlines(
        y=data["date"],
        xmin=data["timestamp_start"].dt.replace(year=2000, month=1, day=1),
        xmax=data["timestamp_end"]
        .clip(
            upper_bound=data["timestamp_start"].dt.date().cast(pl.Datetime)
            + pl.duration(hours=23, minutes=59, seconds=59)
        )
        .dt.replace(year=2000, month=1, day=1),
        color="black",
        linewidths=0.6,
    )

bedtimes = screentime_spans.join(merged_physiologicals, on="date", how="left").filter(
    pl.col("bedtime").ge(pl.col("date").first()),
)
ax.plot(
    bedtimes["bedtime"].dt.replace(year=2000, month=1, day=1),
    bedtimes["bedtime"].dt.date(),
    marker="o",
    color="black",
    markerfacecolor="white",
    markersize=3,
    linestyle="none",
    clip_on=False,
    zorder=1e6,
)
ax.hlines(
    y=bedtimes["bedtime"].dt.date(),
    xmin=bedtimes["bedtime"].dt.replace(year=2000, month=1, day=1),
    xmax=bedtimes["bedtime"].dt.replace(year=2000, month=1, day=1)
    - datetime.timedelta(hours=PRE_BEDTIME_SCREENTIME_WINDOW),
    color="silver",
    linewidths=4,
    zorder=-1e6,
)
bedtime_overflows = (
    bedtimes.filter(
        pl.col("bedtime").dt.date().eq(pl.col("date")),
    )
    .with_columns(
        pl.col("date").sub(pl.duration(days=1)),
        pl.col("bedtime")
        .sub(pl.duration(hours=PRE_BEDTIME_SCREENTIME_WINDOW))
        .alias("clipped_bedtime_window"),
    )
    .filter(pl.col("clipped_bedtime_window").dt.date().eq(pl.col("date")))
    .select(
        "date",
        "timestamp_start",
        "bedtime",
        "clipped_bedtime_window",
    )
)
bedtime_overflows
ax.hlines(
    y=bedtime_overflows["date"],
    xmin=bedtime_overflows["clipped_bedtime_window"].dt.replace(
        year=2000, month=1, day=1
    ),
    xmax=datetime.datetime(year=2000, month=1, day=1) + datetime.timedelta(days=1),
    color="silver",
    linewidths=4,
    zorder=-1e6,
)
ax.xaxis.set_major_formatter(dates.DateFormatter("%H:%M"))
ax.xaxis.set_major_locator(dates.HourLocator(interval=3))
ax.xaxis.set_minor_locator(dates.HourLocator(interval=1))
ax.yaxis.set_major_locator(dates.AutoDateLocator())
ax.yaxis.set_minor_locator(dates.DayLocator(interval=1))
ax.grid(True, axis="y", which="minor")
ax.set_xlim(
    datetime.datetime(year=2000, month=1, day=1),
    datetime.datetime(year=2000, month=1, day=2),
)
ax.set_ylim(
    ymin=pl.select(screentime_spans["date"].last() + pl.duration(days=0.5)),
    ymax=pl.select(screentime_spans["date"].first() - pl.duration(days=0.5)),
)
ax.text(
    0,
    1.05,
    "Date",
    transform=ax.transAxes,
    ha="right",
    va="top",
)
ax.text(
    1.0,
    -0.05,
    "Time of day",
    transform=ax.transAxes,
    ha="right",
    va="top",
)
plt.savefig("paper/artifacts/screentime_slice.png")
plt.close()
# plt.show()

In [13]:
def scatter_correlation_plot(ax, x, y):
    if dimension_x == dimension_y:
        ax.annotate(
            dimension_x,
            xy=(0.5, 0.8),
            xycoords="axes fraction",
            ha="center",
            fontsize=10,
        )
        hist, bin_edges = np.histogram(
            daily_screentime_physiology[x],
            bins=np.linspace(
                daily_screentime_physiology[x].min(),
                daily_screentime_physiology[x].max(),
                11,
            ),
            density=False,
        )
        ax.bar(
            x=bin_edges[:-1],
            height=np.array(hist),
            width=(bin_edges[1] - bin_edges[0]) / 1.2,
            bottom=ax.get_ylim()[0],
            color="none",
            edgecolor="silver",
            align="edge",
        )
        ax.margins(x=0.1, y=0.5)
        return
    ax.plot(
        daily_screentime_physiology[x],
        daily_screentime_physiology[y],
        marker="o",
        markerfacecolor="white",
        linestyle="none",
        clip_on=False,
        zorder=100,
    )
    ax.margins(x=0.1, y=0.1)

    ax.set_xlabel(x)
    ax.set_ylabel(y)


dimensions_of_interest = [
    "screentime_pre_bedtime_hours",
    "sleep_efficiency_percentage",
    "proportion_in_slowwave",
    "duration_asleep_hours",
    "prev_day_strain",
    "bedtime",
    # "Connected with family and/or friends?",
    # "Faced challenges?",
    # "Experienced stress?",
    "prev_morning_hrv",
    "morning_hrv",
]
display(
    daily_screentime_physiology.select(dimensions_of_interest)
    .corr(label="cols")
    .select("cols", pl.selectors.numeric().round(2))
    .pipe(lambda df: df.rename({col: col[:9] for col in df.columns}))
)

fig, ax_rows = plt.subplots(
    figsize=(3 * len(dimensions_of_interest), 3 * len(dimensions_of_interest)),
    nrows=len(dimensions_of_interest),
    ncols=len(dimensions_of_interest),
)
for x_idx, (dimension_x, ax_row) in enumerate(zip(dimensions_of_interest, ax_rows)):
    for y_idx, (dimension_y, ax) in enumerate(zip(dimensions_of_interest, ax_row)):
        if x_idx > y_idx:
            ax.axis("off")
            continue
        scatter_correlation_plot(ax, x=dimension_x, y=dimension_y)

plt.show()

cols,screentim,sleep_eff,proportio,duration_,prev_day_,bedtime,prev_morn,morning_h
str,f64,f64,f64,f64,f64,f64,f64,f64
"""screentime_pre_bedtime_hours""",1.0,-0.21,-0.18,0.02,0.44,-0.35,-0.12,-0.24
"""sleep_efficiency_percentage""",-0.21,1.0,-0.13,0.62,-0.11,-0.01,-0.11,0.29
"""proportion_in_slowwave""",-0.18,-0.13,1.0,-0.22,-0.29,0.31,0.09,-0.04
"""duration_asleep_hours""",0.02,0.62,-0.22,1.0,0.17,-0.18,-0.15,0.3
"""prev_day_strain""",0.44,-0.11,-0.29,0.17,1.0,-0.3,0.11,-0.31
"""bedtime""",-0.35,-0.01,0.31,-0.18,-0.3,1.0,-0.23,-0.22
"""prev_morning_hrv""",-0.12,-0.11,0.09,-0.15,0.11,-0.23,1.0,-0.05
"""morning_hrv""",-0.24,0.29,-0.04,0.3,-0.31,-0.22,-0.05,1.0


In [14]:
def plot_temporal_relationship(
    observations,
    x,
    cause,
    effect,
    cause_label=None,
    effect_label=None,
    cause_color="black",
    effect_color="C2",
    title=None,
):
    fig, ax = plt.subplots(
        figsize=(8, 4.5),
        sharex=True,
    )
    cause_line = ax.plot(
        observations[x],
        observations[cause],
        label=cause_label,
        marker="o",
        markerfacecolor="white",
        linewidth=1,
        color=cause_color,
        clip_on=False,
        zorder=100,
        path_effects=(
            [
                patheffects.Stroke(linewidth=3, foreground="white"),
                patheffects.Normal(),
            ]
        ),
    )[0]
    ax.xaxis.set_major_formatter(dates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(dates.DayLocator(interval=7))
    ax.xaxis.set_minor_locator(dates.DayLocator(interval=1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1_000_000 * 60 * 30))
    ax.yaxis.set_major_formatter(
        lambda val, idx: pl.select(
            pl.duration(microseconds=val).dt.to_string(format="polars")
        ).item()
    )

    ax.set_axisbelow(True)
    ax2 = ax.twinx()
    ax2.set_axisbelow(True)

    ax2.plot(
        observations[x],
        observations[cause],
        marker="o",
        markerfacecolor="white",
        linestyle="none",
        color=cause_color,
        clip_on=False,
        zorder=100,
        path_effects=(
            [
                patheffects.Stroke(linewidth=3, foreground="white"),
                patheffects.Normal(),
            ]
        ),
        transform=ax.transData,
    )[0]

    effect_line = ax2.plot(
        observations[x],
        observations[effect],
        label=effect_label,
        marker="o",
        markerfacecolor="white",
        linewidth=1,
        color=effect_color,
        clip_on=False,
        zorder=100,
        path_effects=(
            [
                patheffects.Stroke(linewidth=3, foreground="white"),
                patheffects.Normal(),
            ]
        ),
    )[0]

    ax.annotate(
        f"{cause_label}",
        xy=(cause_line.get_xdata()[-1], cause_line.get_ydata()[-1].astype(float)),
        # xy=(cause_line.get_xdata()[0], cause_line.get_ydata()[0].astype(float)),
        xytext=(-5, 0),
        textcoords="offset points",
        # arrowprops=dict(
        #     facecolor=cause_line.get_color(),
        #     edgecolor="none",
        #     width=1,
        #     headwidth=4,
        #     headlength=4,
        #     linewidth=0.4,
        # ),
        color=cause_line.get_color(),
        va="center",
        ha="right",
        fontsize=7,
        path_effects=[patheffects.withStroke(linewidth=3, foreground="white")],
        zorder=1e6,
        annotation_clip=False,
    )
    ax2.annotate(
        f"{effect_label}",
        xy=(effect_line.get_xdata()[0], effect_line.get_ydata()[0]),
        xytext=(5, 0),
        textcoords="offset points",
        color=effect_line.get_color(),
        va="center",
        ha="left",
        fontsize=7,
        path_effects=[patheffects.withStroke(linewidth=3, foreground="white")],
        zorder=1e6,
        annotation_clip=False,
    )

    # ax.set_title(title)
    ax.grid(True, axis="y", which="minor")
    ax2.grid(False)

    ax.set_ylim(ymin=1)
    ax.set_ylim(top=1_000_000 * 60 * 60 * PRE_BEDTIME_SCREENTIME_WINDOW)
    ax.margins(x=0)
    ax2.margins(x=0)

    ax2.spines["left"].set_visible(False)
    ax2.spines["right"].set_visible(True)

    plt.savefig("paper/artifacts/time_series.png")
    plt.close()
    # plt.show()


plot_temporal_relationship(
    daily_screentime_physiology,
    title="Pre-bedtime screen time and next-morning HRV",
    x="date",
    cause="screentime_pre_bedtime",
    cause_label=f"Screen time in\n final{' ' + str(PRE_BEDTIME_SCREENTIME_WINDOW) if PRE_BEDTIME_SCREENTIME_WINDOW > 1 else ''} hour{'s' if PRE_BEDTIME_SCREENTIME_WINDOW > 1 else ''}\nprior to bed",
    effect="morning_hrv",
    effect_label="Next-morning\nHRV reading",
)

In [15]:
# data = daily_screentime_physiology.with_columns(
#     pl.col("screentime_pre_bedtime").dt.total_hours(fractional=True),
#     pl.col("duration_asleep").dt.total_hours(fractional=True),
#     pl.col("bedtime").dt.time().cast(pl.Duration).dt.total_hours(fractional=True),
# ).to_pandas()

# model = bmb.Model(
#     data=data,
#     formula="morning_hrv ~ 1 +"
#     + " + ".join(
#         (
#             "screentime_pre_bedtime",
#             # "scale(screentime_pre_bedtime)",
#             # "scale(bedtime)",
#             # "scale(prev_day_strain)",
#             # "scale(sleep_efficiency_percentage)",
#         )
#     ),
#     priors={
#         "Intercept": bmb.Prior("Normal", mu=data["morning_hrv"].mean(), sigma=3),
#         "screentime_pre_bedtime": bmb.Prior("Normal", mu=0, sigma=3),
#         "scale(screentime_pre_bedtime)": bmb.Prior("Normal", mu=0, sigma=2),
#     },
#     family="gaussian",
#     categorical=[],
# )
# print(model)

# fit = model.fit(
#     chains=4,
#     cores=4,
#     tune=500,
#     draws=1000,
#     random_seed=0,
# )

# display(az.summary(fit))
# az.plot_posterior(
#     fit,
#     # var_names=["scale(screentime_pre_bedtime)"],
#     var_names=["screentime_pre_bedtime"],
#     hdi_prob=az.rcParams["stats.ci_prob"],
#     ref_val=0,
# )
# with plt.rc_context(
#     {
#         "axes.prop_cycle": plt.cycler(
#             color=["grey", "black"],
#             linestyle=["dotted", "dotted"],
#         )
#     }
# ):
#     az.plot_ppc(
#         model.predict(fit, kind="response", inplace=False),
#         num_pp_samples=100,
#         figsize=(5, 3.5),
#         show=True,
#     )
# az.plot_trace(fit, show=True)
# plt.show()
# az.plot_khat(
#     az.loo(
#         model.compute_log_likelihood(fit, inplace=False),
#         pointwise=True,
#     )
# )
# plt.margins(x=0.1)

# predictions = bmb.interpret.predictions(
#     model,
#     fit,
#     conditional=dict(
#         # screentime_pre_bedtime=data["screentime_pre_bedtime"],
#         screentime_pre_bedtime=screentime_prediction_range,
#     ),
#     # pps=True,
#     prob=az.rcParams["stats.ci_prob"],
# ).rename(columns=lambda name: name.split("_")[0] if "%" in name else name)

# plt.figure()
# plt.plot(
#     predictions["screentime_pre_bedtime"],
#     predictions["estimate"],
#     marker="o",
#     markerfacecolor="white",
#     clip_on=False,
#     zorder=100,
#     label="Posterior mean",
# )
# plt.fill_between(
#     predictions["screentime_pre_bedtime"],
#     predictions["lower"],
#     predictions["upper"],
#     alpha=0.3,
#     label=f"{az.rcParams["stats.ci_prob"]:.0%} HDI",
# )
# plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.5))
# plt.gca().xaxis.set_major_formatter(ticker.PercentFormatter(2))
# plt.gca().yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
# plt.title("Posterior predictions for next-morning HRV conditional on screen time")
# plt.xlabel(
#     f"Proportion of last{' ' + str(PRE_BEDTIME_SCREENTIME_WINDOW) if PRE_BEDTIME_SCREENTIME_WINDOW > 1 else ''} hour{'s' if PRE_BEDTIME_SCREENTIME_WINDOW > 1 else ''} prior to bed spent on screen time"
# )
# plt.ylabel("Next-morning HRV")
# plt.legend(loc="upper right")
# plt.margins(x=0)
# plt.savefig("paper/artifacts/predictions_naive.png")
# plt.show()

## Data Modeling


In [16]:
def standardize(column):
    return column.sub(column.mean()).truediv(column.std())


data = daily_screentime_physiology.with_columns(
    pl.col("screentime_pre_bedtime").dt.total_hours(fractional=True),
    standardize(
        pl.col("bedtime").dt.time().cast(pl.Duration).dt.total_hours(fractional=True)
    ),
    standardize(pl.col("prev_day_strain")),
    standardize(pl.col("sleep_efficiency_percentage")).alias("sleep_quality"),
    # standardize(pl.col("duration_asleep")).alias("sleep_quality"),
    # standardize(pl.col("slowwave_percentage")).alias("sleep_quality"),
    # standardize(pl.col("morning_hrv")),
    pl.col("morning_hrv"),
).to_pandas()

In [17]:
def check_model_fit(
    trace,
    ppc,
    main_effect="indirect_effect_screen",
    pred_var="morning_hrv",
):
    var_names_of_interest = [
        v
        for v in trace.posterior.data_vars
        if trace.posterior[v].dims == ("chain", "draw")
    ]
    display(
        az.summary(
            trace,
            var_names=var_names_of_interest,
        )
    )
    az.plot_trace(trace, var_names=var_names_of_interest, show=True)
    plt.show()
    for var in main_effect:
        pm.plot_posterior(
            trace,
            var_names=var,
            hdi_prob=az.rcParams["stats.ci_prob"],
            ref_val=0,
        )
        plt.xlim(-5, 3)
        plt.ylim(bottom=0)
        plt.title("")
        plt.savefig("paper/artifacts/posterior.png")
        plt.title(var)
        plt.show()
    with plt.rc_context(
        {
            "axes.prop_cycle": plt.cycler(
                color=["grey", "black"],
                linestyle=["dotted", "dotted"],
            )
        }
    ):
        az.plot_ppc(
            ppc,
            var_names=[pred_var],
            num_pp_samples=100,
            figsize=(5, 3.5),
            show=True,
        )

    pred_samples = trace.posterior[pred_var + "_pred"].values.reshape(
        -1, len(screentime_prediction_range)
    )
    pred_hdi = az.hdi(
        np.expand_dims(pred_samples, 0),
        hdi_prob=az.rcParams["stats.ci_prob"],
    )

    plt.plot(
        screentime_prediction_range,
        pred_samples.mean(axis=0),
        marker="o",
        markerfacecolor="white",
        clip_on=False,
        zorder=100,
        label="Posterior mean",
    )
    plt.fill_between(
        screentime_prediction_range,
        pred_hdi[:, 1],
        pred_hdi[:, 0],
        alpha=0.3,
        label=f"{az.rcParams["stats.ci_prob"]:.0%} HDI",
    )
    plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    plt.gca().xaxis.set_major_formatter(ticker.PercentFormatter(2))
    plt.gca().yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    plt.xlabel(
        f"Proportion of last{' ' + str(PRE_BEDTIME_SCREENTIME_WINDOW) if PRE_BEDTIME_SCREENTIME_WINDOW > 1 else ''} hour{'s' if PRE_BEDTIME_SCREENTIME_WINDOW > 1 else ''} prior to bed spent on screen time"
    )
    plt.ylabel("Next-morning HRV")
    # plt.title("Posterior predictions for HRV conditional on screen time")

    plt.legend(loc="upper right")
    plt.savefig("paper/artifacts/predictions.png")
    plt.show()

In [18]:
# with pm.Model() as simple_model:

#     # Path a: screen_time -> morning_hrv
#     a_screen = pm.Normal("a_screen", mu=0, sigma=3)
#     intercept_hrv = pm.Normal("intercept_hrv", mu=data["morning_hrv"].mean(), sigma=2)
#     sigma_hrv = pm.HalfNormal("sigma_hrv", sigma=5)

#     # Outcome model
#     pm.Normal(
#         "morning_hrv",
#         mu=intercept_hrv + a_screen * data["screentime_pre_bedtime"],
#         sigma=sigma_hrv,
#         observed=data["morning_hrv"],
#     )

#     hrv_pred = pm.Deterministic(
#         "morning_hrv_pred",
#         intercept_hrv + a_screen * screentime_prediction_range,
#     )

#     trace = pm.sample(2000, tune=1000, return_inferencedata=True)
#     ppc = pm.sample_posterior_predictive(trace)


# check_model_fit(trace, ppc, main_effect="a_screen")

In [19]:
# with pm.Model() as mediation_model:

#     # Path a: screen_time -> sleep_quality
#     a = pm.Normal("a", mu=0, sigma=3)
#     intercept_sleep = pm.Normal("intercept_sleep", mu=0, sigma=1)
#     sigma_sleep = pm.HalfNormal("sigma_sleep", sigma=1)

#     # Path b: sleep_quality -> morning_hrv
#     b = pm.Normal("b", mu=0, sigma=3)
#     intercept_hrv = pm.Normal("intercept_hrv", mu=data["morning_hrv"].mean(), sigma=2)
#     sigma_hrv = pm.HalfNormal("sigma_hrv", sigma=10)

#     # Mediator model
#     pm.Normal(
#         "sleep_quality",
#         mu=intercept_sleep + a * data["screentime_pre_bedtime"],
#         sigma=sigma_sleep,
#         observed=data["sleep_quality"],
#     )

#     # Outcome model
#     pm.Normal(
#         "morning_hrv",
#         mu=intercept_hrv + b * data["sleep_quality"],
#         sigma=sigma_hrv,
#         observed=data["morning_hrv"],
#     )

#     # Indirect effect
#     indirect_effect = pm.Deterministic("indirect_effect_screen", a * b)

#     # Predictions across screen time range
#     sleep_pred = pm.Deterministic(
#         "sleep_pred",
#         intercept_sleep + a * screentime_prediction_range,
#     )
#     hrv_pred = pm.Deterministic(
#         "morning_hrv_pred",
#         intercept_hrv + b * sleep_pred,
#     )

#     trace = pm.sample(2000, tune=1000, return_inferencedata=True)
#     ppc = pm.sample_posterior_predictive(trace)


# check_model_fit(trace, ppc)

In [20]:
dag = """
    SCREEN_TIME[Screen time]
    SLEEP_QUALITY[&nbsp;Sleep efficiency&nbsp;]
    BEDTIME[Bedtime consistency]
    NEXT_DAY_HRV[Next-morning HRV]
    STRAIN[Physical strain]
    OTHER_STRESSORS[Other stressors]
    SCREEN_TIME --> SLEEP_QUALITY
    BEDTIME --> SLEEP_QUALITY
    STRAIN --> SLEEP_QUALITY
    OTHER_STRESSORS -.-> SLEEP_QUALITY
    SLEEP_QUALITY --> NEXT_DAY_HRV
    STRAIN --> NEXT_DAY_HRV
    OTHER_STRESSORS -.-> NEXT_DAY_HRV
    classDef unbobserved fill:#fcfcfc,stroke-dasharray:2,stroke:#777777,stroke-width:2px;
    class OTHER_STRESSORS unbobserved;
"""

# dag = """
#     SCREEN_TIME[Screen time]
#     SLEEP_QUALITY[&nbsp;Sleep efficiency&nbsp;]
#     SLEEP_DURATION[&nbsp;Sleep amount&nbsp;]
#     BEDTIME[Bedtime]
#     SWS[Slow-wave sleep]
#     MORNING_HRV[Next-morning HRV]
#     STRAIN[Physical exercise]
#     OTHER_STRESSORS[Other stressors]

#     SCREEN_TIME --> SWS
#     SCREEN_TIME --> SLEEP_DURATION
#     SWS --> MORNING_HRV
#     SLEEP_DURATION --> MORNING_HRV
#     STRAIN --> SWS
#     STRAIN --> SLEEP_DURATION
#     STRAIN --> MORNING_HRV
#     BEDTIME --> SLEEP_DURATION
#     BEDTIME --> SWS
#     STRAIN --> BEDTIME
#     OTHER_STRESSORS -.-> SWS
#     OTHER_STRESSORS -.-> MORNING_HRV
# """

# dag = """
#     SCREEN_TIME[Screen time]
#     SLEEP_QUALITY[&nbsp;Sleep quality&nbsp;]
#     MORNING_HRV[Next-morning HRV]
#     OTHER_STRESSORS[Other stressors]

#     SCREEN_TIME --> SLEEP_QUALITY --> MORNING_HRV
#     OTHER_STRESSORS -.-> SLEEP_QUALITY
#     OTHER_STRESSORS -.-> MORNING_HRV
# """

fig = mermaid.Mermaid(
    str(mermaid.Config(theme=mermaid.configuration.Themes.NEUTRAL))
    + f"""
graph LR
{dag}
"""
    + """classDef unbobserved fill:#fcfcfc,stroke-dasharray:2,stroke:#777777,stroke-width:2px;
class OTHER_STRESSORS unbobserved;""",
    width=1080,
    scale=2,
)
fig.to_png("paper/artifacts/dag.jpeg")
fig

In [21]:
with pm.Model() as full_model:
    # Priors
    a = pm.Normal("a", mu=0, sigma=2)  # screen_time -> sleep_quality
    b = pm.Normal("b", mu=0, sigma=2)  # sleep_quality -> hrv
    c_stress_sleep = pm.Normal("c_stress_sleep", mu=0, sigma=2)
    c_stress_hrv = pm.Normal("c_stress_hrv", mu=0, sigma=2)
    c_bedtime = pm.Normal("c_bedtime", mu=0, sigma=2)
    intercept_sleep = pm.Normal("intercept_sleep", mu=0, sigma=0.5)
    intercept_hrv = pm.Normal("intercept_hrv", mu=data["morning_hrv"].mean(), sigma=2)

    sigma_sleep = pm.HalfNormal("sigma_sleep", sigma=2)
    sigma_hrv = pm.HalfNormal("sigma_hrv", sigma=5)

    # Mediator model
    sleep_quality_obs_var = pm.Normal(
        "sleep_quality",
        mu=(
            intercept_sleep
            + a * data["screentime_pre_bedtime"]
            + c_stress_sleep * data["prev_day_strain"]
            + c_bedtime * data["bedtime"]
        ),
        sigma=sigma_sleep,
        observed=data["sleep_quality"],
    )

    # Outcome model
    hrv_obs_var = pm.Normal(
        "morning_hrv",
        mu=(
            intercept_hrv
            + b * data["sleep_quality"]
            + c_stress_hrv * data["prev_day_strain"]
        ),
        sigma=sigma_hrv,
        observed=data["morning_hrv"],
    )

    # Derived quantities
    indirect_effect_screen = pm.Deterministic("indirect_effect_screen", a * b)
    indirect_effect_strain = pm.Deterministic(
        "indirect_effect_strain", c_stress_sleep * b
    )
    total_effect_strain = pm.Deterministic(
        "total_effect_strain", c_stress_hrv + indirect_effect_strain
    )

    sleep_pred = pm.Deterministic(
        "sleep_pred",
        intercept_sleep
        + a * screentime_prediction_range
        + c_bedtime * data["bedtime"].mean()
        + c_stress_sleep * data["prev_day_strain"].mean(),
    )
    hrv_pred = pm.Deterministic(
        "morning_hrv_pred",
        intercept_hrv + b * sleep_pred + c_stress_hrv * data["prev_day_strain"].mean(),
    )

    trace = pm.sample(2000, tune=1000, return_inferencedata=True)
    ppc = pm.sample_posterior_predictive(trace)

check_model_fit(
    trace,
    ppc,
    main_effect=("total_effect_strain", "indirect_effect_screen"),
)

Initializing NUTS using jitter+adapt_diag...


KeyboardInterrupt: 

In [ ]:
hrv_sd = data["morning_hrv"].std()
screentime_sd = data["screentime_pre_bedtime"].std()
strain_sd = data["prev_day_strain"].std()

screen_d = (
    trace.posterior["indirect_effect_screen"].values.flatten() * screentime_sd / hrv_sd
)
strain_d = trace.posterior["total_effect_strain"].values.flatten() * strain_sd / hrv_sd

print("Effect sizes (Cohen's d)")
print(
    f"  Screen time (mediated) : {screen_d.mean():.3f} (89% CI {az.hdi(screen_d, hdi_prob=0.89).round(3)})"
)
print(
    f"  Physical strain (total): {strain_d.mean():.3f} (89% CI {az.hdi(strain_d, hdi_prob=0.89).round(3)})"
)

Effect sizes (Cohen's d)
  Screen time (mediated) : -0.026 (89% CI [-0.085  0.026])
  Physical strain (total): -0.255 (89% CI [-0.459 -0.048])
